# RAG - FAISS HNSW

## Init

In [2]:
import numpy as np
import faiss
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings

C:\Users\PC\AppData\Local\Temp\ipykernel_14340\1827892903.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


## Configs

In [ ]:
CHUNK_SIZE = 800
CHUNK_OVERLAP = 100
EMBEDDING_MODEL = "BAAI/bge-small-en-v1.5"

## Pipeline

### Document Loader

In [ ]:
def load_document(file_path: str):
    """
    USAGE: documents = load_document(file_path)
    """
    loader = PyPDFLoader(file_path)
    return loader.load()

### Chunking

In [ ]:
def create_chunks(documents):
    """
    USAGE: chunks = create_chunks(documents)
    """
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP
    )
    return splitter.split_documents(documents)

### Embedding

In [ ]:
def embed_chunks(chunks, embedding_model=EMBEDDING_MODEL):
    """
    USAGE: embeddings = embedding_model.encode(chunks)
    """
    texts = [doc.page_content for doc in chunks]
    embedding_model = HuggingFaceEmbeddings(model_name=embedding_model)
    return embedding_model.encode(texts)

### Convert to float32

In [ ]:
def convert_to_float32(embeddings):
    """
    USAGE: embeddings = convert_to_float32(embeddings)
    """
    return np.asarray(embeddings, dtype=np.float32)

### Normalisation

In [ ]:
def normalise(embeddings):
    faiss.normalize_L2(embeddings)
    return embeddings

### Create HNSW

In [ ]:
def create_hnsw(d:int, m=32, efConstruction=200, efSearch=64):
    """
    Input:
        - d: dimension, embeddings.shape[1]
    USAGE: index = create_hnsw(embeddings)
    """
    index = faiss.IndexHNSWFlat(d, m)
    index.hnsw.efConstruction = efConstruction
    index.hnsw.efSearch = efSearch
    return index

### Tune HNSW

### Add Vector

In [ ]:
def add_vectors(index, embeddings):
    index.add(embeddings)

### Save Index (Optional)

In [ ]:
faiss.write_index(
    index,
    "rag_hnsw.index"
)

In [ ]:
index = faiss.read_index(
    "rag_hnsw.index"
)

### Save Chunk Mapping

In [ ]:
documents = [
    {
        "text": "...",
        "source": "...",
        "page": 10
    }
]

### Query

In [ ]:
question = "What is Product Quantization?"

query_embedding = embedding_model.encode(
    [question]
)

query_embedding = np.asarray(
    query_embedding,
    dtype=np.float32
)

faiss.normalize_L2(query_embedding)

### Search

In [ ]:
D, I = index.search(
    query_embedding,
    5
)

### Recover Text

In [ ]:
retrieved

[
    "...PQ explanation...",
    "...Vector Quantization...",
    ...
]

### Build Prompt

In [ ]:
context = "\n\n".join(retrieved)

prompt = f"""
Context:

{context}

Question:

{question}

Answer:
"""

### LLM

In [ ]:
response = llm.invoke(prompt)